# YouTube Auto-Dub — dub + lip-sync on a free Colab GPU

Runtime → **Change runtime type → GPU (T4)**. This dubs a YouTube video into
another language *in the original voice* and (optionally) re-renders the mouth to
match, using Wav2Lip. Everything is open-source and free.

## 1. Check the GPU

In [8]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-f73e2f1a-219c-fbb7-7b48-715f9e679756)


## 2. Install ffmpeg + ytdub
The `[xtts,nllb]` extras pull the cloning TTS and the neural translator. On the
Colab GPU runtime torch already has CUDA, so torchcodec loads fine here.

In [9]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg python3-venv > /dev/null
!ffmpeg -version | head -n 1

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


## 3. Pick a video and dub it (no lip-sync yet)
First run downloads the models (~4 GB). Output lands in `data/output/`.

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'      # accept the XTTS license
URL = 'https://www.youtube.com/shorts/fc4BozJTvc4'  # <-- change me
!ytdub dub "$URL" --target en --asr-model medium --translator nllb

## 4. Set up Wav2Lip (its own venv — its deps conflict with coqui-tts)
We create a separate virtualenv for Wav2Lip and download its checkpoint. If the
checkpoint URL 404s, grab `wav2lip_gan.pth` from the Wav2Lip README and drop it in
`Wav2Lip/checkpoints/`.

In [ ]:
%%bash
set -e
cd /content
[ -d Wav2Lip ] || git clone -q https://github.com/Rudrabha/Wav2Lip
cd Wav2Lip
python3 -m venv .venv
./.venv/bin/pip -q install --upgrade pip
# Wav2Lip pins old libs; these versions are known to work on Colab:
./.venv/bin/pip -q install numpy==1.23.5 opencv-python librosa==0.9.2 numba==0.58.1 \
    torch torchvision tqdm 'scipy<1.13'
mkdir -p checkpoints face_detection/detection/sfd
# face detector + generator checkpoints (swap the URLs if they move):
wget -q -O face_detection/detection/sfd/s3fd.pth \
  https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth || true
wget -q -O checkpoints/wav2lip_gan.pth \
  https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth || true
ls -lh checkpoints/

## 5. Dub **with lip-sync**
Point ytdub at the Wav2Lip venv and re-run with `--lipsync`. Wav2Lip re-renders the
mouth to match the dubbed audio; the result is remuxed as a share-ready MP4.

In [ ]:
from google.colab import files

def is_url(s):
    return str(s).startswith("http://") or str(s).startswith("https://")

video_path = None

if is_url(SOURCE):
    out_tmpl = "/content/source.%(ext)s"
    for old in glob.glob("/content/source.*"):
        os.remove(old)

    cmd = [
        "yt-dlp", "--no-playlist", "--merge-output-format", "mp4",
        "-S", f"res:{MAX_HEIGHT}",
        "-f", f"bv*[height<={MAX_HEIGHT}]+ba/b[height<={MAX_HEIGHT}]/b",
        "--extractor-args", "youtube:player_client=android,web",
        "-o", out_tmpl, SOURCE,
    ]
    if os.path.isfile(COOKIES):
        cmd[1:1] = ["--cookies", COOKIES]
        print("uso cookies:", COOKIES)
    else:
        print("niente cookies.txt — su Colab YouTube spesso fallisce")

    r = run(cmd, check=False)
    found = glob.glob("/content/source.*")
    found = [p for p in found if p.lower().endswith((".mp4", ".mkv", ".webm", ".mov"))]
    if r.returncode != 0 or not found:
        print("\n*** YouTube bloccato (bot-check). Fai UNA di queste:")
        print("  A) carica il video in /content/ e metti SOURCE = '/content/nome.mp4'")
        print("  B) sul PC: estensione 'Get cookies.txt LOCALLY' su youtube.com")
        print("     poi Upload di cookies.txt in /content/cookies.txt e rilancia questa cella")
        print("\nCarica ora un MP4 da questo popup.")
        up = files.upload()
        if not up:
            raise SystemExit("nessun file caricato")
        video_path = "/content/" + list(up.keys())[0]
    else:
        video_path = found[0]
else:
    if not os.path.isfile(SOURCE):
        print(f"{SOURCE} non c'è. Carica il video nel popup (finisce in /content/).")
        up = files.upload()
        if not up:
            raise SystemExit("nessun file caricato")
        video_path = "/content/" + list(up.keys())[0]
    else:
        video_path = SOURCE

video_path = os.path.abspath(video_path)
print("VIDEO PRONTO:", video_path, "size", os.path.getsize(video_path) // (1024*1024), "MB")

## 6. Preview / download the result

In [ ]:
cmd = [
    "ytdub", "dub", video_path,
    "--target", TARGET,
    "--asr-model", ASR_MODEL,
    "--translator", TRANSLATOR,
    "--tts", TTS,
    "--reencode",
]
if BURN_SUBS:
    cmd.append("--subtitles")

run(cmd)

outs = sorted(
    glob.glob("/content/data/output/*.mp4") + glob.glob("data/output/*.mp4"),
    key=os.path.getmtime,
)
if not outs:
    raise SystemExit("nessun mp4 in data/output — guarda l'errore sopra")
dubbed = os.path.abspath(outs[-1])
print("DUB OK:", dubbed)

7

In [ ]:
from IPython.display import Video, display
from google.colab import files

print(dubbed)
display(Video(dubbed, embed=True, width=360))
files.download(dubbed)